# 8.4 Loss 與 Metrics

本 notebook 整理 regression、binary classification 與 multi-class classification 常見 loss/metrics 搭配，並用合成二元分類資料示範 Keras metrics、confusion matrix 與 threshold 調整。


## 1. 載入套件


In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)


## 2. 任務、輸出層、Loss 與 Metrics 對照


In [ ]:
loss_metric_table = pd.DataFrame([
    {
        'task': 'Regression',
        'output_layer': 'Dense(1)',
        'label_format': 'continuous value',
        'loss': 'mse / mae',
        'metrics': 'mae / RMSE',
    },
    {
        'task': 'Binary classification',
        'output_layer': 'Dense(1, sigmoid)',
        'label_format': '0 or 1',
        'loss': 'binary_crossentropy',
        'metrics': 'accuracy / precision / recall / AUC',
    },
    {
        'task': 'Multi-class classification',
        'output_layer': 'Dense(num_classes, softmax)',
        'label_format': 'integer class id',
        'loss': 'sparse_categorical_crossentropy',
        'metrics': 'accuracy',
    },
    {
        'task': 'Multi-class classification',
        'output_layer': 'Dense(num_classes, softmax)',
        'label_format': 'one-hot vector',
        'loss': 'categorical_crossentropy',
        'metrics': 'accuracy',
    },
])
loss_metric_table


## 3. 建立二元分類資料


In [ ]:
X, y = make_classification(
    n_samples=1600,
    n_features=16,
    n_informative=8,
    n_redundant=3,
    weights=[0.68, 0.32],
    class_sep=1.1,
    flip_y=0.03,
    random_state=SEED,
)
X = X.astype('float32')
y = y.astype('float32')

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print(x_train.shape, x_valid.shape, x_test.shape)
print(pd.Series(y_train).value_counts(normalize=True).sort_index())


## 4. 建立模型並設定多個 Metrics


In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc'),
    ],
)

model.summary()


## 5. 訓練與評估


In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_valid, y_valid),
    epochs=25,
    batch_size=32,
    verbose=0,
)

eval_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
eval_result


In [ ]:
history_df = pd.DataFrame(history.history)
history_df[['loss', 'val_loss', 'accuracy', 'val_accuracy', 'precision', 'val_precision', 'recall', 'val_recall']].tail()


## 6. Confusion Matrix 與 Classification Report


In [ ]:
y_prob = model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test.astype(int), y_pred)
print(cm)
print(classification_report(y_test.astype(int), y_pred, digits=3))


## 7. 調整 Threshold 觀察 Precision / Recall 取捨


In [ ]:
threshold_rows = []
for threshold in [0.3, 0.5, 0.7]:
    pred = (y_prob >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
    })

pd.DataFrame(threshold_rows)


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(y_prob[y_test == 0], bins=20, alpha=0.7, label='true class 0')
plt.hist(y_prob[y_test == 1], bins=20, alpha=0.7, label='true class 1')
plt.axvline(0.5, color='black', linestyle='--', label='threshold=0.5')
plt.xlabel('Predicted probability')
plt.ylabel('Count')
plt.title('Predicted Probability Distribution')
plt.legend()
plt.tight_layout()
plt.show()


## 8. 套用自己的資料

先確認任務型態與標籤格式，再決定輸出層、loss 與 metrics。資料不平衡時，不要只看 accuracy；請搭配 precision、recall、AUC 或 confusion matrix。
